# HyDE（Hypothetical Document Embedding）：用“假设文档”拉近查询与文档的分布差

本节目标：实现 HyDE 的最小版本，并理解它为什么能提升检索。

## 核心直觉

向量检索时，一个常见困难是：

- **Query** 往往很短（几句话甚至几个词）
- **文档 chunk** 往往更长、更具体

这会导致“短 query 的向量分布”和“长 chunk 的向量分布”不完全对齐。

HyDE 的做法是：

1. 给定用户问题 `query`
2. 用 LLM 先生成一个**假设文档**（hypothetical document）：它像是一段“直接回答该问题的正文”
3. 用这个 hypothetical document 去做向量检索（而不是用原始 query）

这样做的直观好处是：检索时你用的文本形态更像 chunk（更长、更具体），更容易和语料 chunk 在同一语义空间对齐。

## 0) 导入依赖并加载环境变量

延续前面 notebook 的约定：从 `../.env` 读取 `OPENAI_API_KEY`。

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import sys

from dotenv import load_dotenv

load_dotenv("../.env")

# 让 `all_rag_techniques/` 子目录里的 notebook
# 也能导入上一级目录的 `helper_functions.py`
sys.path.insert(0, str(Path("..").resolve()))

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from helper_functions import show_context

/tmp/ipykernel_3229226/2061229951.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1) 准备文档路径

跟原仓库一样，我们使用同一份气候变化 PDF。你如果已经在前面章节跑过，这个文件路径通常已经可用。

In [2]:
path = Path("../data/Understanding_Climate_Change.pdf")
assert path.exists(), f"找不到 PDF: {path.resolve()}"

path

PosixPath('../data/Understanding_Climate_Change.pdf')

## 2) 构建向量库（chunks → embeddings → FAISS）

这里我们故意采用较小的 chunk（默认 500）来贴近原 notebook：

- `chunk_size=500`
- `chunk_overlap=100`

HyDE 里这个参数还有一层含义：**hypothetical document 的长度最好接近 chunk 的长度**，这样检索向量会更“同分布”。

In [3]:
def encode_pdf_to_faiss(
    pdf_path: Path,
    *,
    chunk_size: int,
    chunk_overlap: int,
):
    docs = PyPDFLoader(str(pdf_path)).load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    splits = splitter.split_documents(docs)

    vectorstore = FAISS.from_documents(
        documents=splits,
        embedding=OpenAIEmbeddings(),
    )

    return vectorstore, splits


CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

vectorstore, splits = encode_pdf_to_faiss(
    path,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

len(splits)

201

## 3) 定义 HyDE 检索器

我们实现一个最小 `HyDERetriever`，它做三件事：

- 生成 hypothetical document
- 用 hypothetical document 做向量检索
- 返回：检索到的 chunks + hypothetical document（方便你观察）

In [4]:
@dataclass
class HyDERetriever:
    vectorstore: FAISS
    chunk_size: int = 500

    def __post_init__(self) -> None:
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

        self.prompt_hyde = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "你是一个用于 HyDE 的 hypothetical document 生成器。\n"
                    "给定问题，请直接写出一段‘像文档正文一样’的回答段落，用于向量检索。\n"
                    "要求：\n"
                    "- 内容要具体、信息密度高\n"
                    "- 尽量覆盖问题涉及的关键点\n"
                    "- 只输出正文，不要标题/列表前缀/解释\n"
                    "- 字符长度尽量接近 {chunk_size}（允许略有偏差）"
                ),
                ("user", "问题：{query}\n\nhypothetical document："),
            ]
        )
        self.chain_hyde = self.prompt_hyde | self.llm

    def generate_hypothetical_document(self, query: str) -> str:
        return self.chain_hyde.invoke({"query": query, "chunk_size": self.chunk_size}).content.strip()

    def retrieve(self, query: str, *, k: int = 3) -> Tuple[List, str]:
        hypothetical_doc = self.generate_hypothetical_document(query)
        docs = self.vectorstore.similarity_search(hypothetical_doc, k=k)
        return docs, hypothetical_doc

## 4) 演示：生成 hypothetical document + 检索 top-k chunks

我们会打印两部分：

1. hypothetical document（你可以观察它是否“像 chunk 一样具体”）
2. 检索出来的 chunks（看看是否更容易命中答案所在段落）

In [5]:
retriever = HyDERetriever(vectorstore=vectorstore, chunk_size=CHUNK_SIZE)

test_query = "What is the main cause of climate change?"
results, hypothetical_doc = retriever.retrieve(test_query, k=3)

docs_content = [d.page_content for d in results]

print("hypothetical_doc:\n")
print(hypothetical_doc)
print("\n" + "=" * 80 + "\n")
show_context(docs_content)

hypothetical_doc:

气候变化的主要原因是温室气体的排放，尤其是二氧化碳、甲烷和氧化亚氮等。这些气体主要来源于人类活动，如燃烧化石燃料（煤、石油和天然气）用于发电、交通和工业生产，导致大气中温室气体浓度显著增加。森林砍伐和土地利用变化也加剧了这一问题，因为树木吸收二氧化碳，减少森林覆盖会导致更多的二氧化碳留在大气中。此外，农业活动，特别是牲畜的饲养和化肥的使用，释放大量甲烷和氧化亚氮，这些气体的温室效应强度远高于二氧化碳。工业过程、废物处理和城市化进程同样贡献了温室气体的排放。气候变化的影响已经显现，包括全球气温上升、极端天气事件频发、海平面上升以及生态系统的破坏，威胁到人类的生存和发展。因此，减缓气候变化的关键在于减少温室气体的排放，转向可再生能源、提高能效、保护和恢复森林以及推广可持续的农业实践，以实现全球气候目标。


Context 1:
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked

Context 2:
predict future trends. The evidence overwhelmingly shows that recent changes are primarily 
driven by human activities, particularly the